In [ ]:
# 1/4 - clone CUDA backend branch (HEAD c75925b: layernorm.tkv + for DSL)
import subprocess, os
d='/content/TokenVectorCuda'
subprocess.run('rm -rf '+d, shell=True)
r=subprocess.run('git clone --depth 1 --branch prototype/cuda-backend https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
print('HEAD', subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())

In [ ]:
# 2/4 - build .so from layernorm.cuda.cu (compiler-generated)
import subprocess, shutil, os
ex='/content/TokenVectorCuda/Release/cuda_backend/examples'
cu=ex+'/layernorm.cuda.cu'; so=ex+'/layernorm.so'
nvcc=shutil.which('nvcc') or '/usr/local/cuda/bin/nvcc'
cmd=(nvcc+' -O2 -shared -std=c++20 --expt-relaxed-constexpr -x cu -Xcompiler -std=c++20 -Xcompiler -fPIC '+cu+' -o '+so+' -lcudart')
p=subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=600)
print('build rc=',p.returncode)
if p.returncode!=0: print(p.stderr[:3000])
print('so exists',os.path.exists(so))

In [ ]:
# 3/4 - test LayerNorm .tkv vs torch + speed
import ctypes, torch, time
so='/content/TokenVectorCuda/Release/cuda_backend/examples/layernorm.so'
lib=ctypes.CDLL(so); I32=ctypes.c_int; I64=ctypes.c_int64; F32=ctypes.c_float
lib.tkv_launch_layernorm_fwd.argtypes=[I64,I64,I64,I64,I32,I32,F32,I32,I32]; lib.tkv_launch_layernorm_fwd.restype=I32
rows=4096; h=768
x=torch.randn(rows,h,device='cuda',dtype=torch.float32)
g=torch.randn(h,device='cuda',dtype=torch.float32); b=torch.randn(h,device='cuda',dtype=torch.float32)
out=torch.empty_like(x)
rc=lib.tkv_launch_layernorm_fwd(x.data_ptr(),g.data_ptr(),b.data_ptr(),out.data_ptr(),rows,h,1e-6,rows,256)
ref=torch.nn.functional.layer_norm(x,(h,),g,b,1e-6)
err=(out-ref).abs().max().item()
print('layernorm rc=',rc,' maxerr=',err,' CORRECT=',err<1e-3)
# speed
for _ in range(3): lib.tkv_launch_layernorm_fwd(x.data_ptr(),g.data_ptr(),b.data_ptr(),out.data_ptr(),rows,h,1e-6,rows,256)
t0=time.perf_counter()
for _ in range(20): lib.tkv_launch_layernorm_fwd(x.data_ptr(),g.data_ptr(),b.data_ptr(),out.data_ptr(),rows,h,1e-6,rows,256)
t1=time.perf_counter()
t0b=time.perf_counter()
for _ in range(20): ref=torch.nn.functional.layer_norm(x,(h,),g,b,1e-6)
t1b=time.perf_counter()
print('TKV ln %.3f ms  torch ln %.3f ms  ratio %.2fx'%((t1-t0)/20*1000,(t1b-t0b)/20*1000,(t1b-t0b)/(t1-t0)))

In [ ]:
# 4/4 - summary
print('LayerNorm .tkv test done. If CORRECT=True and ratio>1, kernel faster than ATen.')